# MasriCX — resumable Kaggle GPU runner

Thin orchestration shell only. **No training logic lives here**; all research logic remains in `src/masricx/`.

## Goal

Clone the authorized public repository, install dependencies, read `HF_TOKEN` from Kaggle Secrets, and invoke the tested resumable training entry point.

### Key assumptions

- Kaggle GPU and internet are enabled.
- `HF_TOKEN` exists in Kaggle Secrets and is never printed.
- The orchestrator supplies `REPO_URL` and authorizes the remote run.
- Publishing checkpoints or model artifacts is a separate orchestrator-owned action.


In [ ]:
import os
import subprocess
from pathlib import Path

from kaggle_secrets import UserSecretsClient

REPO_URL = "https://github.com/tarekamr737/MasriCX"
EXPECTED_COMMIT = None  # Orchestrator must set the pushed 40-character Git commit.
CONFIG = "configs/pilot.yaml"
PILOT_RUN = "P1"  # P1/P2/P3, or None for E1/E2
MAX_STEPS = None  # Set to 2 for a fresh numerical smoke run.
CHECKPOINT_REPO = None  # Set only after private Hub repo creation is authorized.
WORKSPACE = Path("/tmp/MasriCX")
OUTPUT_DIR = Path("/kaggle/working/masricx-training")

assert REPO_URL.startswith("https://github.com/"), "Set the authorized REPO_URL"
assert isinstance(EXPECTED_COMMIT, str) and len(EXPECTED_COMMIT) == 40, "Pin EXPECTED_COMMIT"

try:
    hf_token = UserSecretsClient().get_secret("HF_TOKEN")
except Exception:  # Secret is optional for public model downloads.
    hf_token = None
if hf_token:
    os.environ["HF_TOKEN"] = hf_token
os.environ["HF_HOME"] = "/tmp/huggingface"

if not WORKSPACE.exists():
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(WORKSPACE)], check=True)
actual_commit = subprocess.run(
    ["git", "rev-parse", "HEAD"], cwd=WORKSPACE, check=True, capture_output=True, text=True
).stdout.strip()
assert actual_commit == EXPECTED_COMMIT, (actual_commit, EXPECTED_COMMIT)
subprocess.run(
    ["python", "-m", "pip", "install", "-r", str(WORKSPACE / "requirements-training.txt")],
    check=True,
)
subprocess.run(["python", "-m", "pip", "install", "-e", str(WORKSPACE), "--no-deps"], check=True)

if PILOT_RUN:
    command = [
        "python",
        "-m",
        "masricx.training.pilot",
        "--config",
        CONFIG,
        "--run",
        PILOT_RUN,
        "--output-dir",
        str(OUTPUT_DIR),
        "--split-dir",
        "data/splits",
        "--auto-resume",
        "--execute",
    ]
else:
    command = [
        "python",
        "-m",
        "masricx.training.train",
        "--config",
        CONFIG,
        "--output-dir",
        str(OUTPUT_DIR),
        "--split-dir",
        "data/splits",
        "--auto-resume",
        "--execute",
    ]
if CHECKPOINT_REPO:
    command.extend(["--checkpoint-repo", CHECKPOINT_REPO])
if MAX_STEPS is not None:
    command.extend(["--max-steps", str(MAX_STEPS)])
subprocess.run(command, cwd=WORKSPACE, check=True)

adapter_configs = list(OUTPUT_DIR.rglob("adapter_config.json"))
adapter_weights = list(OUTPUT_DIR.rglob("adapter_model.safetensors")) + list(
    OUTPUT_DIR.rglob("adapter_model.bin")
)
assert adapter_configs and adapter_weights, "Training did not produce a complete adapter"
print({"adapter_configs": len(adapter_configs), "adapter_weights": len(adapter_weights)})